# Segunda Parte del Proyecto

## Carga de Datasets 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
#Acceso a Internet Oficiales
dfInternetOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/1L__Instituciones_Oficiales_Internet.csv",
    header=True, inferSchema=True
)
dfInternetOficiales.createOrReplaceTempView("dfInternetOficiales")
display(dfInternetOficiales)

In [0]:
dfInternetOficiales.printSchema()

In [0]:
#Acceso a Internet No Oficiales
dfInternetNoOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/2L__Instituciones_No_Oficiales_Internet.csv",
    header=True, inferSchema=True
)
dfInternetNoOficiales.createOrReplaceTempView("dfInternetNoOficiales")
display(dfInternetNoOficiales)

In [0]:

#Matricula por Municipio
dfMatricula = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/3L__Matrícula_Por_Municipio.csv",
    header=True, inferSchema=True
)
dfMatricula.createOrReplaceTempView("dfMatricula")
display(dfMatricula)

In [0]:


#Estadísticas Básica-Media
dfEstadisticas = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/4L__Estadística_Básica_Media.csv",
    header=True, inferSchema=True
)
dfEstadisticas.createOrReplaceTempView("dfEstadisticas")
display(dfEstadisticas)

In [0]:


#Estrategia UNIDOS
dfUNIDOS = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/5L__Estrategia_UNIDOS.csv",
    header=True, inferSchema=True
)
dfUNIDOS.createOrReplaceTempView("dfUNIDOS")
display(dfUNIDOS)

In [0]:


#Internet Fijo
dfInternet = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/6L__Internet_Fijo_Tecnología_Segmento.csv",
    header=True, inferSchema=True
)
dfInternet.createOrReplaceTempView("dfInternet")
display(dfInternet)

In [0]:

#Pruebas Saber
dfSaber = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/7L__Resultados_Únicos_Saber_11.csv",
    header=True, inferSchema=True
)
dfSaber.createOrReplaceTempView("dfSaber")
display(dfSaber)

## Filtros y Transformaciones

Con la intención de comenzar a plantear los Joins, primero se realiza la estandarización de los municipios por df por mayúsculas y sin carácteres especiales. Se deja en la columna "municipio_std"

In [0]:
dfInternetOficiales = dfInternetOficiales.withColumn("municipio_std",upper(trim(F.col("municipio"))))

dfInternetOficiales = dfInternetOficiales.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfInternetOficiales.select("municipio_std").distinct().show(150, False)

In [0]:
dfInternetNoOficiales = dfInternetNoOficiales.withColumn("municipio_std",upper(trim(F.col("municipio"))))

dfInternetNoOficiales = dfInternetNoOficiales.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfInternetNoOficiales.select("municipio_std").distinct().show(150, False)

In [0]:
dfMatricula = dfMatricula.withColumn("municipio_std",upper(trim(F.col("municipio"))))

dfMatricula = dfMatricula.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfMatricula.select("municipio_std").distinct().show(150, False)

In [0]:
dfEstadisticas = dfEstadisticas.withColumn("municipio_std",upper(trim(F.col("municipio"))))

dfEstadisticas = dfEstadisticas.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfEstadisticas.select("municipio_std").distinct().show(150, False)

In [0]:
dfUNIDOS = dfUNIDOS.withColumn("municipio_std",upper(trim(F.col("NombreMunicipioAtencion"))))

dfUNIDOS = dfUNIDOS.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfUNIDOS.select("municipio_std").distinct().show(150, False)

In [0]:
dfInternet = dfInternet.withColumn("municipio_std",upper(trim(F.col("MUNICIPIO"))))

dfInternet = dfInternet.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfInternet.select("municipio_std").distinct().show(150, False)

In [0]:
dfSaber = dfSaber.withColumn("municipio_std",upper(trim(F.col("COLE_MCPIO_UBICACION"))))

dfSaber = dfSaber.withColumn("municipio_std",translate(F.col("municipio_std"),"ÁÉÍÓÚÄËÏÖÜ","AEIOUAEIOU"))

dfSaber.select("municipio_std").distinct().show(150, False)


##### Filtro #1
Antes se había observado que en el dfSaber habían algunas filas con municipio como "BOYACA" Por lo cual se deciden eliminar esas entradas ya que no nos dan información valiosa al no tener un municipio centralizado.

In [0]:
dfSaber = dfSaber.filter(upper(trim(col("municipio_std"))) != "BOYACA")

dfSaber.select("municipio_std").distinct().show(100, False)

### Agregación por Dataset

Se realiza agregación por municipio para después comenzar a unirlos en el dataframe al cual se le va a aplicar el modelo de machine learning

#### InternetOficiales

##### Transformación #1
Se propone solamente tomar la velocidad, muncipio y ancho de banda para este dataset y el de InternetNoOficiales con tal de hacer un join entre ambos y llegar a conclusiones sobre la conectividad.

In [0]:
dfInternetOficiales.printSchema()

Primero volvemos valores numéricos a los anchos de banda

In [0]:
dfInternetOficiales = dfInternetOficiales.withColumn("ancho_subida",F.when(F.col("ancho-de-banda-de-subida") == "null", None).otherwise(F.col("ancho-de-banda-de-subida")).cast("double"))

dfInternetOficiales = dfInternetOficiales.withColumn("ancho_bajada",F.when(F.col("ancho-de-banda-descarga") == "null", None).otherwise(F.col("ancho-de-banda-descarga")).cast("double"))

dfInternetOficiales.select("ancho_subida", "ancho_bajada").printSchema()

Después hacemos la agregación por municipio para ver los promedios de velocidad

In [0]:
dfInternetOficiales = dfInternetOficiales.withColumn("tiene_ancho_banda",F.col("tiene_ancho_banda").cast("double"))

Se propone crear la métrica de porcentajes de sedes que tienen internet de colegios oficiales, el indicador de calidad del internet y de conectividad que es la cobertura + calidad

In [0]:
dfInternetOficialesAGG = dfInternetOficiales.groupBy("municipio_std").agg(
    F.avg("ancho_subida").alias("avg_subida"),
    F.avg("ancho_bajada").alias("avg_bajada"),
    F.count("*").alias("total_sedes"),
    F.sum("tiene_ancho_banda").alias("sedes_con_internet")
)

In [0]:
dfInternetOficiales.select("ancho-de-banda-de-subida").distinct().show(20, False)

- pct_sedes_internet es la proporción de escuelas que tienen internet
- avg_velocidad es que tan buena es la velocidad realmente
- indice_conectividad es las dos anteriores sumadas 

In [0]:

dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("pct_sedes_internet",F.col("sedes_con_internet") / F.col("total_sedes"))
dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("avg_velocidad",(F.col("avg_subida") + F.col("avg_bajada")) / 2)
dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("indice_conectividad",F.col("avg_velocidad") * F.col("pct_sedes_internet"))

dfInternetOficialesAGG.orderBy("pct_sedes_internet").show(10)

Adicionalmente se planea sacar un índice de desconexión, que indicaria las zonas con menor acceso a internet de colegios oficiales por lo que: 
- pct_sin_internet son los que no tienen ancho de banda

In [0]:
dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("pct_sin_internet",1 - F.col("pct_sedes_internet"))

dfInternetOficialesAGG.select("municipio_std","pct_sedes_internet","pct_sin_internet").show(10)

Se procede a estandarizar la velocidad

In [0]:
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn(
    "velocidad_norm",
    (F.col("avg_velocidad") - F.min("avg_velocidad").over(w)) /
    (F.max("avg_velocidad").over(w) - F.min("avg_velocidad").over(w))
)

dfInternetOficialesAGG.select("municipio_std","avg_velocidad","velocidad_norm").show(10)

Y se procede a mejorar el índice de conectividad anteriormente planteado

In [0]:
dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("indice_conectividad_v2",F.col("velocidad_norm") * F.col("pct_sedes_internet"))
dfInternetOficialesAGG.select("municipio_std","pct_sedes_internet","velocidad_norm","indice_conectividad_v2").show(10)

#### InternetNoOficiales

Siguiendo con la idea anterior, se propone realizar los mismos índices y unir ambos dataframes

In [0]:
dfInternetNoOficiales.printSchema()

In [0]:
#Convertir a double el acnho de banda
dfInternetNoOficiales = dfInternetNoOficiales.withColumn("ancho_banda",
    F.when((F.col("ancho-de-banda-num") == "null") | (F.col("ancho-de-banda-num") == ""),None).otherwise(F.col("ancho-de-banda-num").cast("double")))
dfInternetNoOficiales.select("ancho-de-banda-num", "ancho_banda").show(5)

In [0]:
#Se crea el mismo indicador de velocidad como acá no tenemos de subida y de bajada
dfInternetNoOficiales = dfInternetNoOficiales.withColumn("velocidad",F.col("ancho_banda"))

Se tiene en cuenta que acá hay sub-representación de ciertos municipios, sobretodo porque este es un dataframe considerablemente más pequeño que el anterior

In [0]:
#Se hace la misma agregación de municipios 
dfInternetNoOficialesAGG = dfInternetNoOficiales.groupBy("municipio_std").agg(
    F.avg("velocidad").alias("avg_velocidad"),
    F.count("*").alias("total_sedes"),
    F.sum("tiene-ancho-banda").alias("sedes_con_internet"))

dfInternetNoOficialesAGG.show(10)

In [0]:
#Cálculo de la cobertura y falta de acceso
dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("pct_sedes_internet",F.col("sedes_con_internet") / F.col("total_sedes"))
dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("pct_sin_internet",1 - F.col("pct_sedes_internet"))

In [0]:
#Normalización de la velocidad (esta vaina es adelantándonos a lo de ML)
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("velocidad_norm",(F.col("avg_velocidad") - F.min("avg_velocidad").over(w)) /(F.max("avg_velocidad").over(w) - F.min("avg_velocidad").over(w)))

In [0]:
#Cálculo del índice de conectividad
dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("indice_conectividad_v2",F.col("velocidad_norm") * F.col("pct_sedes_internet"))

dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("velocidad_norm",F.when(F.col("velocidad_norm").isNull(), 0).otherwise(F.col("velocidad_norm")))

dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("indice_conectividad_v2",
    F.when(F.col("indice_conectividad_v2").isNull(), 0).otherwise(F.col("indice_conectividad_v2")))

En los datos de instituciones no oficiales se observa una baja variabilidad en la velocidad de conexión, concentrándose principalmente en valores mínimos. Esto limita la capacidad del índice para diferenciar la calidad del servicio entre municipios.

In [0]:
dfInternetNoOficialesAGG.orderBy("indice_conectividad_v2").show()

##### Transformación #2

Se propone unir los dos anteriores dataset por medio del municipio con tal de identificar la diferencias entre sector privado y público respecto a la cobertura

In [0]:
df_pub = dfInternetOficialesAGG.select("municipio_std",F.col("indice_conectividad_v2").alias("indice_publico"),F.col("pct_sedes_internet").alias("pct_publico"),F.col("avg_velocidad").alias("velocidad_publico"))

df_priv = dfInternetNoOficialesAGG.select("municipio_std",F.col("indice_conectividad_v2").alias("indice_privado"),F.col("pct_sedes_internet").alias("pct_privado"),F.col("avg_velocidad").alias("velocidad_privado"))

dfFinal = df_pub.join(
    df_priv,
    on="municipio_std",
    how="inner"
)

dfFinal.show(150)